# Whisper v2 与 v3 同 50 条对比

In [1]:
import csv
import gc
import re
import time
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from IPython.display import display

from peft import PeftModel
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
)

### 定义相关参数

In [9]:
# datasets 路径
CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

NUM_SAMPLES = 200
SAMPLING_RATE = 16000
MAX_NEW_TOKENS = 100
NUM_BEAMS = 1
DEVICE = "cuda"

# 原始 v2/v3 对比时保持 None。
# v2 微调完成后，填入 adapter 目录即可。
V2_ADAPTER_DIR = "models/whisper-large-v2-asr-int8"
# V2_ADAPTER_DIR = None
# v2 基座模型
V2_MODEL_ID = "openai/whisper-large-v2"

# 如果以后也训练了 v3，可填写 v3 adapter 路径。
V3_ADAPTER_DIR = "/sigan/ai-ml/models/whisper-large-v3-zh-safe-evalfix"
# V3_ADAPTER_DIR = None
# v3 基座模型
V3_MODEL_ID = "openai/whisper-large-v3"

MODEL_SPECS = [
    {
        "name": "Whisper-v2",
        "model_id": V2_MODEL_ID,
        "adapter_dir": V2_ADAPTER_DIR,
    },
    {
        "name": "Whisper-v3",
        "model_id": V3_MODEL_ID,
        "adapter_dir": V3_ADAPTER_DIR,
    },
]

### 读取处理数据

#### 定义归一函数

In [10]:
import re
import unicodedata

import opencc

# 繁体转简体
_t2s = opencc.OpenCC("t2s")

# 统一常见中英文标点
_PUNCT_MAP = str.maketrans({
    ",": "，",
    "?": "？",
    "!": "！",
    ":": "：",
    ";": "；",
    "(": "（",
    ")": "）",
    "[": "【",
    "]": "】",
    "{": "｛",
    "}": "｝",
    "、": "，",
})

# 简繁转换与标点归一化函数
def normalize_chinese(text, remove_punctuation=False):
    """
    Chinese ASR 文本统一：
    1. 繁体 -> 简体
    2. 全角字符标准化
    3. 英文转小写
    4. 删除空格
    5. 统一常见标点
    6. 可选删除全部标点
    """
    if text is None:
        return ""

    # 先统一繁简体
    text = _t2s.convert(text)

    # 将全角字母、数字等统一为半角形式
    text = unicodedata.normalize("NFKC", text)

    # 英文大小写统一
    text = text.lower()

    # 中文 ASR 通常不依赖空格
    text = re.sub(r"\s+", "", text)

    # 英文句点不误伤小数，例如 3.14 保持不变
    text = re.sub(r"(?<!\d)\.(?!\d)", "。", text)

    # 统一中英文标点
    text = text.translate(_PUNCT_MAP)

    # 连续标点压缩成一个
    text = re.sub(r"，+", "，", text)
    text = re.sub(r"。+", "。", text)
    text = re.sub(r"？+", "？", text)
    text = re.sub(r"！+", "！", text)

    # 如果只比较识别内容，可以删除全部标点
    if remove_punctuation:
        text = re.sub(
            r"[，。？！：；、（）【】｛｝,\.\?!:;()\[\]{}]",
            "",
            text,
        )

    return text.strip()

In [11]:
# 读取相同 50 条测试数据
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    rows = list(
        csv.DictReader(handle, delimiter="\t")
    )[:NUM_SAMPLES]

references = []
audio_arrays = []

# 音频只读取一次，两个模型使用完全相同的波形
for index, row in enumerate(rows, start=1):
    audio_path = CV_ROOT / "clips" / row["path"]

    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLING_RATE,
    )

    # references.append(row["sentence"])
    # 标准答案也使用同一套归一化规则
    references.append(normalize_chinese(row["sentence"],remove_punctuation=True))
    audio_arrays.append(audio)

    if index % 10 == 0:
        print(f"音频已加载：{index}/{NUM_SAMPLES}")

# 使用更完善的归一函数
# def clean_text(text):
#     # 去掉不同模型可能插入的空格
#     return re.sub(r"\s+", "", text).strip()


def run_model(model_id, adapter_dir):
    # v2 使用 80 Mel，v3 使用 128 Mel，所以每个模型加载自己的 Processor
    processor = AutoProcessor.from_pretrained(
        model_id,
        language="Chinese",
        task="transcribe",
    )

    # 强制 Whisper 使用中文转写
    forced_decoder_ids = (
        processor.get_decoder_prompt_ids(
            language="chinese",
            task="transcribe",
        )
    )

    # 8-bit 加载，确保两个模型可以分别在 16GB 显存上运行
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        quantization_config=BitsAndBytesConfig(
            load_in_8bit=True
        ),
        device_map="auto",
    )

    base_model.config.use_cache = True
    model = base_model

    # 可选：挂载训练后的 LoRA adapter
    if adapter_dir:
        model = PeftModel.from_pretrained(
            base_model,
            adapter_dir,
        )

    model.eval()

    # 不要重复传 forced_decoder_ids，否则可能出现参数冲突
    model.generation_config.forced_decoder_ids = (
        forced_decoder_ids
    )

    predictions = []
    start_time = time.perf_counter()

    for index, audio in enumerate(audio_arrays, start=1):
        inputs = processor(
            audio,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
        ).to(DEVICE)

        # 8-bit 模型的卷积层使用 FP16
        inputs["input_features"] = (
            inputs["input_features"].half()
        )

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )

        prediction = processor.batch_decode(
            output_ids,
            skip_special_tokens=True,
        )[0]

        predictions.append(normalize_chinese(prediction,remove_punctuation=True))

        if index % 5 == 0:
            print(f"已生成：{index}/{NUM_SAMPLES}")

    elapsed = time.perf_counter() - start_time

    # 及时释放显存，准备加载下一个模型
    del model
    del base_model
    del processor
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, elapsed

音频已加载：10/200
音频已加载：20/200
音频已加载：30/200
音频已加载：40/200
音频已加载：50/200
音频已加载：60/200
音频已加载：70/200
音频已加载：80/200
音频已加载：90/200
音频已加载：100/200
音频已加载：110/200
音频已加载：120/200
音频已加载：130/200
音频已加载：140/200
音频已加载：150/200
音频已加载：160/200
音频已加载：170/200
音频已加载：180/200
音频已加载：190/200
音频已加载：200/200


### 开始测试数据

In [12]:
all_results = {}

In [13]:
for spec in MODEL_SPECS:
    print("\n开始测试：", spec["name"])

    predictions, elapsed = run_model(
        spec["model_id"],
        spec["adapter_dir"],
    )

    overall_cer = jiwer.cer(
        reference=references,
        hypothesis=predictions,
    )

    all_results[spec["name"]] = {
        "predictions": predictions,
        "cer": overall_cer,
        "runtime": elapsed,
    }

    print(
        spec["name"],
        "总体 CER：",
        overall_cer,
        "耗时：",
        round(elapsed, 2),
        "秒",
    )


开始测试： Whisper-v2


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


已生成：5/200
已生成：10/200
已生成：15/200
已生成：20/200
已生成：25/200
已生成：30/200
已生成：35/200
已生成：40/200
已生成：45/200
已生成：50/200
已生成：55/200
已生成：60/200
已生成：65/200
已生成：70/200
已生成：75/200
已生成：80/200
已生成：85/200
已生成：90/200
已生成：95/200
已生成：100/200
已生成：105/200
已生成：110/200
已生成：115/200
已生成：120/200
已生成：125/200
已生成：130/200
已生成：135/200
已生成：140/200
已生成：145/200
已生成：150/200
已生成：155/200
已生成：160/200
已生成：165/200
已生成：170/200
已生成：175/200
已生成：180/200
已生成：185/200
已生成：190/200
已生成：195/200
已生成：200/200
Whisper-v2 总体 CER： 0.09161976235146967 耗时： 509.05 秒

开始测试： Whisper-v3


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


已生成：5/200
已生成：10/200
已生成：15/200
已生成：20/200
已生成：25/200
已生成：30/200
已生成：35/200
已生成：40/200
已生成：45/200
已生成：50/200
已生成：55/200
已生成：60/200
已生成：65/200
已生成：70/200
已生成：75/200
已生成：80/200
已生成：85/200
已生成：90/200
已生成：95/200
已生成：100/200
已生成：105/200
已生成：110/200
已生成：115/200
已生成：120/200
已生成：125/200
已生成：130/200
已生成：135/200
已生成：140/200
已生成：145/200
已生成：150/200
已生成：155/200
已生成：160/200
已生成：165/200
已生成：170/200
已生成：175/200
已生成：180/200
已生成：185/200
已生成：190/200
已生成：195/200
已生成：200/200
Whisper-v3 总体 CER： 0.13070669168230145 耗时： 493.12 秒


### 对比数据

In [14]:
# 生成逐条对比表
v2 = all_results["Whisper-v2"]["predictions"]
v3 = all_results["Whisper-v3"]["predictions"]

table_rows = []

for index, reference in enumerate(references, start=1):
    v2_cer = jiwer.cer(
        reference=reference,
        hypothesis=v2[index - 1],
    )

    v3_cer = jiwer.cer(
        reference=reference,
        hypothesis=v3[index - 1],
    )

    table_rows.append({
        "序号": index,
        "标准答案": reference,
        "v2 输出": v2[index - 1],
        "v3 输出": v3[index - 1],
        "v2 CER": round(v2_cer, 4),
        "v3 CER": round(v3_cer, 4),
        # 正数表示 v3 比 v2 更好
        "v3改善": round(v2_cer - v3_cer, 4),
    })

comparison_df = pd.DataFrame(table_rows)
comparison_df = comparison_df.sort_values(
    by="v3改善",
    ascending=False,
)

display(comparison_df)

v3_better = (comparison_df["v3改善"] > 0).sum()
equal_count = (comparison_df["v3改善"] == 0).sum()
v3_worse = (comparison_df["v3改善"] < 0).sum()

,序号,标准答案,v2 输出,v3 输出,v2 CER,v3 CER,v3改善
99,100,乖孙啊,关孙昂,乖孙老,0.6667,0.3333,0.3333
68,69,宋朝因之,宋朝殷居,宋朝英之,0.5000,0.2500,0.2500
144,145,嫡母程氏生母苏氏,比姆城市生物书室,离母尘世生母疏逝,0.8750,0.6250,0.2500
178,179,邪马台国在哪里,协马台国在哪里,邪马台国在哪里,0.1429,0.0000,0.1429
172,173,向量组的秩即为向量组中的任一最大线性无关组所含有的向量个数,项量组的质即为项量组中任意最大泄洗无关组所含有的项量个数,向量组的质即为向量组中任意最大线系无关组所含有的向量个数,0.2759,0.1379,0.1379
...,...,...,...,...,...,...,...
42,43,该区首府为卡永,该区首府为卡永,该屈手俯为卡拥,0.0000,0.5714,-0.5714
184,185,则以及如上,则以及儒尚,这一集入场,0.4000,1.0000,-0.6000
60,61,芒德维尔,芒德韦尔,忙得为耳,0.2500,1.0000,-0.7500
193,194,雅夫雷扎克,亚夫雷扎克,有什么问题,0.2000,1.0000,-0.8000


### 总结

In [15]:
print("\n总体对比")
print("v2 CER：", all_results["Whisper-v2"]["cer"])
print("v3 CER：", all_results["Whisper-v3"]["cer"])
print("v3 更好：", v3_better)
print("两者相同：", equal_count)
print("v3 更差：", v3_worse)


总体对比
v2 CER： 0.09161976235146967
v3 CER： 0.13070669168230145
v3 更好： 30
两者相同： 95
v3 更差： 75
